In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False
os.makedirs('figures', exist_ok=True)


def style(ax, title, xlabel, ylabel):
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
    ax.legend()


# ===== 练习 1：复指数信号与 I/Q 分解 =====
print("练习1：复指数信号 exp(j·2πf0·t) 的实部(I)与虚部(Q)")
fs = 1000                      # 采样率 Hz
t1 = np.arange(0, 0.2, 1/fs)   # 0.2 秒
f0 = 50                        # 信号频率 Hz
x1 = np.exp(1j * 2 * np.pi * f0 * t1)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(t1, x1.real, label='I (实部 cos)')
ax[0].plot(t1, x1.imag, label='Q (虚部 sin)', linestyle='--')
style(ax[0], '复指数信号的 I/Q 两路', '时间 (s)', '幅度')
ax[1].plot(x1.real, x1.imag, label='轨迹')
ax[1].set_aspect('equal')
style(ax[1], '复平面轨迹：单位圆', 'I', 'Q')
plt.tight_layout()
plt.savefig('figures/w1_1_complex.png', dpi=120)
plt.show()
print("验收：I 是 cos、Q 是 sin，相位差 90°；复平面是一个圆\n")


# ===== 练习 2：傅里叶变换看频谱 =====
print("练习2：三个频率叠加的信号，用 FFT 看频谱")
t2 = np.arange(0, 1, 1/fs)
f_sig = np.array([50, 120, 200])          # 三个频率成分
x2 = (np.sin(2*np.pi*f_sig[0]*t2) +
      0.5*np.sin(2*np.pi*f_sig[1]*t2) +
      0.25*np.sin(2*np.pi*f_sig[2]*t2))

N = len(x2)
X = np.fft.fft(x2)
freqs = np.fft.fftfreq(N, 1/fs)
pos = freqs[:N//2]                        # 只取正频率
mag = np.abs(X[:N//2]) * 2 / N

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(t2[:200], x2[:200])
style(ax[0], '时域：三个正弦叠加', '时间 (s)', '幅度')
ax[1].plot(pos, mag)
ax[1].set_xlim(0, 300)
ax[1].set_xticks(f_sig)
style(ax[1], '频域：三个峰值对应三个频率', '频率 (Hz)', '幅度')
plt.tight_layout()
plt.savefig('figures/w1_2_fft.png', dpi=120)
plt.show()
print(f"验收：频谱在 {list(f_sig)} Hz 处出现三个峰\n")


# ===== 练习 3：采样定理与混叠 =====
print("练习3：采样率不足会发生混叠（aliasing）")
f_signal = 80                             # 信号频率
t_cont = np.arange(0, 0.05, 1/10000)      # 近似连续
x_cont = np.sin(2*np.pi*f_signal*t_cont)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(t_cont*1000, x_cont, label='连续信号 80Hz')
# 采样率 200Hz (>2×80) —— 满足奈奎斯特
t_ok = np.arange(0, 0.05, 1/200)
ax[0].stem(t_ok*1000, np.sin(2*np.pi*f_signal*t_ok),
           linefmt='C1-', markerfmt='C1o', basefmt=' ', label='200Hz 采样（够）')
style(ax[0], '采样率足够：能还原原信号', '时间 (ms)', '幅度')
# 采样率 100Hz (<2×80) —— 混叠
t_bad = np.arange(0, 0.05, 1/100)
x_bad = np.sin(2*np.pi*f_signal*t_bad)
ax[1].plot(t_cont*1000, x_cont, label='原信号 80Hz', alpha=0.4)
ax[1].stem(t_bad*1000, x_bad, linefmt='C3-', markerfmt='C3o', basefmt=' ', label='100Hz 采样（不够）')
style(ax[1], '采样率不足：混叠成 20Hz 假信号', '时间 (ms)', '幅度')
plt.tight_layout()
plt.savefig('figures/w1_3_aliasing.png', dpi=120)
plt.show()
print("验收：右图采样点连成的低频波形（20Hz）是假象——这就是混叠\n")


# ===== 练习 4：卷积与低通滤波 =====
print("练习4：卷积即滤波——方波通过低通滤波器变成圆滑波形")
fs4 = 1000
t4 = np.arange(0, 0.1, 1/fs4)
square = signal.square(2*np.pi*20*t4)     # 20Hz 方波
# 4 阶巴特沃斯低通，截止 60Hz
b, a = signal.butter(4, 60/(fs4/2), 'low')
filtered = signal.filtfilt(b, a, square)  # 零相位滤波

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(t4*1000, square, label='方波（含高频谐波）')
style(ax[0], '滤波前：时域方波', '时间 (ms)', '幅度')
ax[1].plot(t4*1000, filtered, color='C2', label='低通滤波后')
style(ax[1], '滤波后：高频被滤掉，边沿变圆滑', '时间 (ms)', '幅度')
plt.tight_layout()
plt.savefig('figures/w1_4_filter.png', dpi=120)
plt.show()
print("验收：能解释『低通滤波 = 卷积一个 sinc 类冲激响应』\n")

print("W1 全部完成。四张图保存在 figures/ 目录。")
